# 05. Visualizaciones interactivas y animaciones orbitales

Este cuaderno incorpora controles interactivos y animaciones a la biblioteca `bohr_atom`.

## Objetivos

- Explorar niveles de energía y transiciones mediante controles de `ipywidgets`.
- Seleccionar el nivel principal y la fase angular del electrón.
- Reproducir una animación de la coordenada relativa electrón–núcleo.
- Examinar la velocidad angular y el período orbital derivados del modelo.

> **Interpretación física:** la animación representa la órbita circular postulada por el modelo histórico de Bohr. No representa una trayectoria observable del electrón en la mecánica cuántica moderna. La coordenada radial es la separación relativa electrón–núcleo, coherente con la masa reducida. La reproducción está acelerada y no utiliza el período atómico real como tiempo de pantalla.

> **Convención de documentación:** cada celda de código comienza con un comentario en español que explica su propósito. Las ecuaciones y los supuestos físicos se explican en las celdas Markdown; la implementación de los cálculos permanece en el paquete `bohr_atom`, no se duplica en este notebook.


## 0. Instalación de dependencias

Desde la raíz del repositorio, activa el entorno virtual e instala el extra de notebooks:

```bash
python -m pip install -e ".[notebooks]"
```

Este extra instala JupyterLab, `ipywidgets` e `ipympl`. Matplotlib ya forma parte de las dependencias principales. `ipympl` proporciona un lienzo Matplotlib actualizable mediante widgets.

In [ ]:
# Activa el backend ipympl para que el lienzo de Matplotlib responda a los widgets.
%matplotlib widget


In [ ]:
# Importa las herramientas de visualización, animación y los sistemas físicos de ejemplo.
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from bohr_atom import (
    BohrAtom,
    PROTON_MASS_KG,
    HELIUM4_NUCLEUS_MASS_KG,
)
from bohr_atom.interactive import (
    interactive_energy_diagram,
    interactive_orbit_viewer,
    animate_orbit,
)

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "font.size": 10,
})

## 1. Crear los sistemas que vamos a explorar

Se utilizarán el hidrógeno y el ion He⁺. Ambos son sistemas hidrogenoides, pero tienen diferente carga nuclear y masa nuclear.

In [ ]:
# Construye los objetos hidrogenoides que se utilizarán en los controles interactivos.
hydrogen = BohrAtom(
    Z=1,
    nucleus_mass_kg=PROTON_MASS_KG,
    name="Hidrógeno"
)

helium_ion = BohrAtom(
    Z=2,
    nucleus_mass_kg=HELIUM4_NUCLEUS_MASS_KG,
    name="He⁺"
)

print(hydrogen)
print(helium_ion)

## 2. Explorador interactivo de niveles de energía

Los controles permiten modificar el número de niveles dibujados y elegir un nivel inicial y otro final. El diagrama y las propiedades del fotón se actualizan al mover los controles.

- Si `n_inicial > n_final`, el proceso es emisión.
- Si `n_inicial < n_final`, el proceso es absorción.
- Los dos niveles deben ser distintos y estar incluidos en el diagrama.

In [ ]:
# Muestra el explorador interactivo de niveles y transiciones para el hidrógeno.
display(interactive_energy_diagram(hydrogen, max_n=10))

### 2.1. Comparación interactiva con He⁺

Repite el análisis para otro sistema hidrogenoide. Observa cómo cambia la escala energética al cambiar `Z`.

In [ ]:
# Explora los niveles y las transiciones del ion He+ mediante controles interactivos.
display(interactive_energy_diagram(helium_ion, max_n=10))

## 3. Visor interactivo de la órbita

Este visor incorpora:

- Un selector del número cuántico principal `n`.
- Un control de reproducción (`Play`).
- Un deslizador de fase angular vinculado al control de reproducción.
- Un control de velocidad visual (sin modificar la velocidad física calculada).
- Casillas para mostrar u ocultar la trayectoria y el vector tangente.
- Información de radio, velocidad angular y período orbital.

El visor mantiene un único lienzo Matplotlib y actualiza sus elementos gráficos; para ello requiere el backend `ipympl`.

El control de reproducción modifica la fase angular de la representación. No es una simulación temporal a escala real.

In [ ]:
# Abre el visor de órbitas: permite elegir n, mover la fase y reproducir la vuelta.
display(interactive_orbit_viewer(
    hydrogen,
    n_max=6,
    frames=120,
    interval=20
))

## 4. Animación orbital con Matplotlib

Además del visor controlado por widgets, podemos generar un archivo/representación de animación reproducible mediante `matplotlib.animation.FuncAnimation`.

El objeto de animación se convierte a HTML/JavaScript para que pueda reproducirse dentro del notebook. Es importante conservar la referencia `animation_n2`: de lo contrario, Python podría eliminar el objeto y detener la animación.

### Ajustes de rendimiento

Las coordenadas se precalculan y la estela se limita mediante `trail_length`. Para un objetivo de 50 FPS se recomienda `interval=20` y, al exportar a HTML, `to_jshtml(fps=50)`. En 2D se utiliza blitting cuando el backend lo admite. En 3D Matplotlib debe redibujar la escena completa, así que el rendimiento puede ser inferior. El FPS real depende del backend y del equipo.


In [ ]:
# Construye la animación de una vuelta visual del electrón en el nivel n=2.
animation_n2 = animate_orbit(
    atom=hydrogen,
    n=2,
    frames=120,
    interval=20,
    revolutions=1,
    show_trail=True,
    trail_length=48,
)

display(HTML(animation_n2.to_jshtml(fps=50)))

## 5. Comparar animaciones para diferentes niveles

El radio crece como `n²`, mientras que la velocidad angular orbital decrece como `1/n³`. La siguiente tabla permite examinar esas tendencias.

In [ ]:
# Calcula radio, velocidad angular y período orbital para los primeros niveles.
import pandas as pd

orbital_data = []
for n in range(1, 6):
    orbital_data.append({
        "n": n,
        "radio (pm)": hydrogen.radius(n, unit="pm"),
        "velocidad angular (rad/s)": hydrogen.orbital_angular_frequency(n),
        "período orbital (s)": hydrogen.orbital_period(n),
    })

pd.DataFrame(orbital_data)

### 5.1. Animación de otro nivel

Cambia `n=2` por otro valor permitido. Por ejemplo, para `n=3`:

In [ ]:
# Construye una segunda animación para comparar el tamaño de la órbita n=3.
animation_n3 = animate_orbit(
    atom=hydrogen,
    n=3,
    frames=120,
    interval=20,
    revolutions=1,
    show_trail=True,
    trail_length=48,
)

display(HTML(animation_n3.to_jshtml(fps=50)))

## 6. Fundamento de la velocidad angular orbital

La velocidad angular se obtiene mediante `ω = v/r`. Para el movimiento relativo con masa reducida `μ`, el modelo de Bohr conduce a:

\[
\omega_n = \frac{\mu Z^2 e^4}
{(4\pi\varepsilon_0)^2\hbar^3 n^3}
\]

El período orbital correspondiente es:

\[
T_n = \frac{2\pi}{\omega_n}
\]

Por tanto, para un mismo sistema:

\[
\omega_n\propto n^{-3},\qquad T_n\propto n^3.
\]

La duración de la animación en pantalla se controla mediante `interval` y `frames`. No debe confundirse con `T_n`, que es el período físico predicho por el modelo.

## 7. Guardar una animación como archivo HTML

La representación JavaScript se puede guardar y abrir en un navegador. El archivo incluye los fotogramas y los controles de reproducción.

In [ ]:
# Exporta la animación anterior a un archivo HTML reproducible fuera del notebook.
html_animation = animation_n2.to_jshtml(fps=50)

with open("orbita_hidrogeno_n2.html", "w", encoding="utf-8") as file:
    file.write(html_animation)

print("Animación guardada en orbita_hidrogeno_n2.html")

## 8. Ejercicios

1. Compara el radio y la velocidad angular de H y He⁺ para `n=1` y `n=3`.
2. Comprueba numéricamente que `ω(n=2)/ω(n=1) = 1/8`.
3. Comprueba que `T(n=2)/T(n=1) = 8`.
4. Utiliza el explorador de energía para representar una emisión `4 → 2` y una absorción `2 → 4`.
5. Explica por qué la animación no debe interpretarse como una órbita real de la mecánica cuántica.
6. Modifica `revolutions`, `frames` e `interval` y analiza cómo cambia la reproducción sin alterar el período físico calculado.

## 9. Visualización tridimensional de un átomo

La siguiente visualización representa la coordenada relativa electrón–núcleo en un sistema de ejes 3D. La órbita de Bohr sigue siendo una circunferencia plana; el parámetro de inclinación permite orientar ese plano en el espacio y la cámara se puede rotar con el lienzo interactivo.

La rapidez que se muestra es la rapidez tangencial relativa, calculada como $v_{\m rel}=\omega r$. También se muestra la velocidad angular física $\omega$ en rad/s. No debe confundirse con la velocidad de reproducción de la animación, que se acelera para hacerla visible.

In [ ]:
# Dibuja una vista 3D estática del nivel n=2 del hidrógeno.
figura_3d, ejes_3d = hydrogen.plot_orbit_3d(
    n=2,
    phase=np.pi / 4,
    inclination_deg=35,
    elev=25,
    azim=45,
    show_velocity=True,
)
plt.show()

### 9.1. Visor 3D interactivo

El visor permite elegir el nivel cuántico, reproducir o pausar la rotación, controlar la fase, inclinar el plano orbital y cambiar la orientación de la cámara. El panel informa radio, velocidad angular, frecuencia orbital, rapidez relativa y período físico.

In [ ]:
# Abre el visor 3D interactivo para el átomo de hidrógeno.
display(
    hydrogen.interactive_orbit_3d_viewer(
        n_max=6,
        frames=120,
        interval=20,
    )
)

### 9.2. Comparar otro sistema hidrogenoide

La misma función recibe cualquier instancia de `BohrAtom`. En este ejemplo se visualiza He⁺: el cambio de carga nuclear modifica el radio, la velocidad angular y la rapidez relativa.

In [ ]:
# Abre el mismo visor para el ion He+.
display(helium_ion.interactive_orbit_3d_viewer(n_max=6, frames=120))

### 9.3. Animación 3D exportable

Además del visor que responde a widgets, `animate_orbit_3d` genera un objeto `FuncAnimation` que puede convertirse a HTML y guardarse fuera del notebook.

### Ajustes de rendimiento

Las coordenadas se precalculan y la estela se limita mediante `trail_length`. Para un objetivo de 50 FPS se recomienda `interval=20` y, al exportar a HTML, `to_jshtml(fps=50)`. En 2D se utiliza blitting cuando el backend lo admite. En 3D Matplotlib debe redibujar la escena completa, así que el rendimiento puede ser inferior. El FPS real depende del backend y del equipo.


In [ ]:
# Crea y muestra una animación 3D de una vuelta en el nivel n=2.
animation_3d = hydrogen.animate_orbit_3d(
    n=2,
    frames=120,
    interval=20,
    revolutions=1,
    inclination_deg=35,
    show_trail=True,
    trail_length=48,
)
display(HTML(animation_3d.to_jshtml(fps=50)))

**Interpretación física:** el radio dibujado es la separación relativa electrón–núcleo, coherente con el uso de masa reducida. La rapidez $v_{\rm rel}$ corresponde a la derivada temporal de esa coordenada relativa. En la aproximación de dos cuerpos, no es exactamente la rapidez del electrón respecto del centro de masa; esta última requiere multiplicar por $M/(M+m_e)$. La representación 3D sigue siendo una órbita clásica de Bohr y no una función de onda.